# USGS Water — site discovery, statistics, and Parquet

USGS Water Data exposes the national network of stream gauges as tables, so `earthlens` treats `usgs-water` as a **tabular** backend. This notebook enumerates the gauges in a small bounding box, then pulls a monthly statistical summary for one of them and writes it out as Parquet.

> `usgs-water` is an anonymous, public backend — no credentials are needed.

## Setup

A single import: `EarthLens`, the unified entry point. Each request below names `data_source="usgs-water"` and a `service` (`sites` or `statistics`).

In [ ]:
from earthlens.earthlens import EarthLens

## 1 · Discover the gauges in a bounding box

The `sites` service returns one row per gauge inside the area of interest. We build the request first — source, variable (`discharge`), date window, bounding box, and the output path — keeping construction and download on separate lines so each step is easy to read and re-run.

In [ ]:
sites_query = EarthLens(
    data_source="usgs-water",
    variables=["discharge"],
    start="2023-01-01",
    end="2023-01-05",
    aoi=[-77.3, 38.8, -76.9, 39.1],
    path="_nb_out",
    service="sites",
    api="legacy",
)
sites = sites_query.download(progress_bar=False)

### The gauges found

`download()` returns a `DataFrame`, one row per gauge. A handful of identifying columns — site number, station name, and location — is enough to see what was discovered.

In [ ]:
sites[["site_no", "station_name", "latitude", "longitude"]].head(10)

## 2 · Monthly statistics for one gauge

Switching to the `statistics` service and naming a single `sites=` gauge pulls its long-term record. Here we ask for a `monthly` statistical summary over four years and request `parquet` output. Again, the request is constructed first, then downloaded.

In [ ]:
stats_query = EarthLens(
    data_source="usgs-water",
    variables=["discharge"],
    start="2018-01-01",
    end="2021-12-31",
    aoi=[-77.2, 38.9, -77.0, 39.0],
    path="_nb_out",
    service="statistics",
    sites="01646500",
    api="legacy",
    stat_type="monthly",
    output_format="parquet",
)
stats = stats_query.download(progress_bar=False)

### The monthly summary

The result is a tidy `DataFrame` of per-month statistics for the gauge, which was also written to `_nb_out` as a Parquet file. The first few rows show its shape.

In [ ]:
stats.head()